<a href="https://colab.research.google.com/github/Sageh9/MSSP607/blob/main/week10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
from pathlib import Path

USER_AGENT_WIKIMEDIA = "StudentProjectBot/1.0 (https://example.com)"
USER_AGENT_DOWNLOAD = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
REFERER_WIKIMEDIA = "https://commons.wikimedia.org/"

def fetch_image_pages(term, limit):
    """
    Fetches image page information from Wikimedia Commons based on a search term.

    Args:
        term (str): The search term for images.
        limit (int): The maximum number of image pages to fetch.

    Returns:
        list: A list of tuples, each containing (title, image_url).
    """
    url = "https://commons.wikimedia.org/w/api.php"
    headers = {"User-Agent": USER_AGENT_WIKIMEDIA}
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": term,
        "gsrlimit": str(limit),
        "gsrnamespace": "6",
        "prop": "imageinfo",
        "iiprop": "url",
        "format": "json"
    }
    r = requests.get(url, params=params, headers=headers, timeout=20)
    r.raise_for_status()
    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    results = []
    for pid, p in pages.items():
        iinfo = p.get("imageinfo")
        if not iinfo:
            continue
        imgurl = iinfo[0].get("url")
        title = p.get("title")
        if imgurl:
            results.append((title, imgurl))
    return results

def download_images(results, outdir):
    """
    Downloads images from a list of URLs to a specified output directory.

    Args:
        results (list): A list of tuples, each containing (title, image_url).
        outdir (str): The directory where images will be saved.
    """
    output_path = Path(outdir)
    output_path.mkdir(parents=True, exist_ok=True)

    for idx, (title, url) in enumerate(results, 1):
        ext = Path(url.split("?")[0]).suffix
        if not ext:
            ext = ".jpg"

        # Sanitize filename: remove 'File:', replace '/' with '_', and ensure valid chars
        clean_title = title.replace('File:', '').replace('/', '_')
        safe_name = "".join(c if c.isalnum() or c in "_-. " else '_' for c in clean_title)
        fname = f"{idx:03d}_{safe_name}{ext}"
        path = output_path / fname

        headers = {
            "User-Agent": USER_AGENT_DOWNLOAD,
            "Referer": REFERER_WIKIMEDIA
        }
        try:
            with requests.get(url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
            print(f"Saved {path}")
        except Exception as e:
            print(f"Failed {url} -> {e}")

def main():
    """
    Main function to orchestrate image fetching and downloading.
    """
    term = input("Enter a keyword to search for images: ")

    limit_input = input("How many images to download? (default 10): ")
    if not limit_input.strip():
        limit = 10
    else:
        try:
            limit = int(limit_input)
            if limit <= 0:
                print("Please enter a positive number for the limit. Defaulting to 10.")
                limit = 10
        except ValueError:
            print("Invalid input for limit. Please enter a number. Defaulting to 10.")
            limit = 10

    results = fetch_image_pages(term, limit)
    if not results:
        print("No images found.")
        return

    outdir = f"images_{term.replace(' ','_')}"
    download_images(results, outdir)
    print("Done.")

if __name__ == "__main__":
    main()


Enter a keyword to search for images: cat
How many images to download? (default 10): 10
Saved images_cat/001_20231125 housecat south meadows PD100306.jpg.jpg
Saved images_cat/002_Cat November 2010-1a.jpg.jpg
Saved images_cat/003_Cat Sphynx. Kittens. img 11.jpg.jpg
Saved images_cat/004_Cat playing with a lizard.jpg.jpg
Saved images_cat/005_Felis catus-cat on snow.jpg.jpg
Saved images_cat/006_June odd-eyed-cat cropped.jpg.jpg
Saved images_cat/007_Six weeks old cat _aka_.jpg.jpg
Saved images_cat/008_Sleeping cat on her back.jpg.jpg
Saved images_cat/009_Tabby cat with blue eyes-3336579.jpg.jpg
Saved images_cat/010_Tired 20-year-old cat.jpg.jpg
Done.


## Q:
make recommendations on improving the Python code.

## A

I've reviewed your Python code and have some recommendations for improvements. I'll make the following changes:

Input Validation: Added better error handling for the limit input to ensure it's a valid integer.
Path Handling: Replaced os.path functions with the more modern and object-oriented pathlib module.
Constants: Defined the User-Agent headers as constants to avoid repetition.
Docstrings: Added basic docstrings to functions for clarity.
Main Execution Guard: Wrapped the main() function call in if __name__ == "__main__": for better module reusability.
Filename Sanitization: Improved filename sanitization to allow more common characters while preventing issues.
Here is the updated code:

The code failed with a SyntaxError. It looks like there's a missing closing brace and quote in the f-string on line 66. I'll correct that now.

Great news! The improved code ran successfully. It fetched and downloaded 10 images of 'cat' into the images_cat directory. You can now find the downloaded images in your file system.